# Mercari Price Suggestion Challenge
## Phase II — Model Training and Evaluation

### Project Purpose
Mercari is Japan's largest community-powered shopping app, where individuals sell new and used items. A key challenge for sellers is determining the right price for their listings. This project builds a machine learning model that **automatically suggests a fair product price** based on the listing's text description, category, brand, condition, and shipping information.

### Objectives
- Train and evaluate two regression models: **Ridge Regression** (linear baseline) and **LightGBM** (gradient boosting)
- Compare model performance across two train-validation splits: **70/30** and **80/20**
- Select the best-performing model for deployment in Phase III
- Evaluate using **RMSLE** (Root Mean Squared Logarithmic Error) — the official Kaggle competition metric

### Input
Feature matrix `X` and target vector `y` produced by Phase I (Feature Engineering), stored as `.pkl` files.

### Output
Trained model files saved to `models/` for use in the FastAPI deployment (Phase III).

---
## Step 1 — Import Libraries

We import all necessary libraries for model training and evaluation:
- `numpy`, `pandas` — numerical operations and data handling
- `joblib` — loading large `.pkl` files (feature matrix, encoders, models)
- `Ridge` — scikit-learn's Ridge Regression (L2-regularized linear model)
- `LGBMRegressor` — LightGBM's gradient boosting regression model
- `train_test_split` — splitting data into training and validation sets
- `mean_squared_error` — base metric for computing RMSLE

In [1]:
import numpy as np
import pandas as pd
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor

print("All libraries loaded ✅")

All libraries loaded ✅


---
## Step 2 — Load Feature Matrix and Target Vector

We load the pre-built feature matrix `X` and target vector `y` from Phase I (Feature Engineering).

- `X` is a **sparse matrix** of shape `(1,481,661 × 100,009)` — each row is one product listing represented as 100,009 numeric features (TF-IDF on name and description + label-encoded categories + numeric features)
- `y` contains the **log1p-transformed price** (`log_price`) for each listing — this is what the model learns to predict

We use `log_price` as the target (instead of raw price) because:
1. Price is heavily right-skewed — log transformation normalizes the distribution
2. RMSLE (our evaluation metric) is mathematically equivalent to RMSE on log-transformed targets

In [2]:
# Load Features and Target

print("Loading features and target...")

X = joblib.load('../data/processed/X_features.pkl')
y = joblib.load('../data/processed/y_target.pkl')

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"y range : {y.min():.3f} to {y.max():.3f}")
print(f"\nX type  : {type(X)}")

Loading features and target...
X shape : (1481661, 100009)
y shape : (1481661,)
y range : 1.386 to 7.606

X type  : <class 'scipy.sparse._csr.csr_matrix'>


---
## Step 3 — Evaluation Metric Functions

We define two helper functions:

**`rmsle(y_true, y_pred)`** — Computes RMSLE (Root Mean Squared Logarithmic Error). Since our target `y` is already `log1p(price)`, RMSLE reduces to plain RMSE on these log-transformed values. Lower RMSLE = better model.

**`evaluate(model_name, y_true, y_pred)`** — Prints a formatted evaluation report including:
- RMSLE — the primary competition metric
- MAE in USD — converts predictions back to real prices using `np.expm1()` for human-interpretable error (e.g. "model is off by $10.44 on average")

**`train_and_compare_splits(model, model_name, X, y)`** — Trains a given model on both 70/30 and 80/20 splits and returns results for comparison. `random_state=42` ensures the same split every run, making results reproducible.

In [4]:
# RMSLE Function

def rmsle(y_true, y_pred):
    """
    Calculate RMSLE.
    Since target is already log1p transformed,
    RMSLE = RMSE of log-transformed values.
    """
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate(model_name, y_true, y_pred):
    """Print evaluation metrics for a model."""
    score = rmsle(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Convert back to original price scale for interpretability
    y_true_price = np.expm1(y_true)
    y_pred_price = np.expm1(y_pred)
    mae_price    = np.mean(np.abs(y_true_price - y_pred_price))

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  RMSLE          : {score:.4f}")
    print(f"  RMSE (log)     : {rmse:.4f}")
    print(f"  MAE (USD)      : ${mae_price:.2f}  ← real price scale")
    print(f"{'='*45}")
    
    return score

## Step 4 — Split Comparison Function


In [ ]:
# Split Comparison Function

def train_and_compare_splits(model, model_name, X, y):
    """
    Train model on 70/30 and 80/20 splits.
    Returns best split result.
    """
    results = {}

    for test_size, split_name in [(0.30, '70/30'), (0.20, '80/20')]:
        
        print(f"\nTraining {model_name} | Split: {split_name}")
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, y,
            test_size=test_size,
            random_state=42    #   for reproducibility 
        )
        
        print(f"  Train size : {X_train.shape[0]:,}")
        print(f"  Val size   : {X_val.shape[0]:,}")
        
        start = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start
        
        y_pred = model.predict(X_val)
        
        # Clip predictions — negative log_price
        y_pred = np.clip(y_pred, 0, None)
        
        score = evaluate(f"{model_name} | {split_name}", y_val, y_pred)
        print(f"  Train time : {train_time:.1f}s")
        
        results[split_name] = {
            'model'     : model,
            'score'     : score,
            'train_time': train_time,
            'X_train'   : X_train,
            'X_val'     : X_val,
            'y_train'   : y_train,
            'y_val'     : y_val,
            'y_pred'    : y_pred
        }
    
    # Better split (lower RMSLE = better)
    best_split = min(results, key=lambda k: results[k]['score'])
    print(f"\n  ✅ Best split for {model_name}: {best_split}")
    
    return results, best_split

---
## Step-5: Model 1 — Ridge Regression

### Why Ridge Regression?

Ridge Regression is a **linear model** with L2 regularization. It was selected as the **baseline model** for the following reasons:

1. **Sparse matrix compatibility** — Our feature matrix is 99% sparse (TF-IDF). Ridge with `solver='sparse_cg'` (Conjugate Gradient) handles this efficiently without dense matrix conversion.
2. **Speed** — Ridge trains in under 2 seconds on 1.4 million rows, making it ideal for rapid iteration and benchmarking.
3. **Regularization** — `alpha=10` penalizes large coefficients, reducing overfitting on the high-dimensional (100,009 features) TF-IDF space.
4. **Text data strength** — Linear models perform well on TF-IDF features because each word's contribution to price is largely additive (e.g. "iPhone" adds value, "used" subtracts it).

**Limitation:** Ridge assumes a linear relationship between features and price. It cannot capture interaction effects — for example, that a "Nike" brand item in "New" condition commands a disproportionately higher premium than either feature alone would suggest.

**Key hyperparameters:**
- `alpha=10` — regularization strength (higher = simpler model, less overfitting)
- `solver='sparse_cg'` — optimized solver for sparse input matrices
- `max_iter=100` — maximum iterations for convergence

In [7]:
# Train Ridge Regression

print("=" * 50)
print("MODEL 1: RIDGE REGRESSION")
print("=" * 50)

ridge_model = Ridge(alpha=10, solver='sparse_cg', max_iter=100)

ridge_results, ridge_best_split = train_and_compare_splits(
    model=ridge_model,
    model_name="Ridge Regression",
    X=X,
    y=y
)

MODEL 1: RIDGE REGRESSION

Training Ridge Regression | Split: 70/30
  Train size : 1,037,162
  Val size   : 444,499

  Ridge Regression | 70/30
  RMSLE          : 0.6246
  RMSE (log)     : 0.6246
  MAE (USD)      : $14.17  ← real price scale
  Train time : 1.8s

Training Ridge Regression | Split: 80/20
  Train size : 1,185,328
  Val size   : 296,333

  Ridge Regression | 80/20
  RMSLE          : 0.6248
  RMSE (log)     : 0.6248
  MAE (USD)      : $14.16  ← real price scale
  Train time : 2.1s

  ✅ Best split for Ridge Regression: 70/30


---
## Step-6: Model 2 — LightGBM (Light Gradient Boosting Machine)

### Why LightGBM?

LightGBM is a **gradient boosting framework** that builds an ensemble of decision trees sequentially, where each tree corrects the errors of the previous one. It was selected as the **primary model** for the following reasons:

1. **Non-linear pattern capture** — Unlike Ridge, LightGBM captures complex interaction effects. For example, it can learn that `brand=Nike` + `condition=New` + `category=Shoes` together command a price premium far beyond the sum of individual features.
2. **Industry standard for tabular data** — LightGBM consistently outperforms linear models on structured/tabular datasets. It placed in the top solutions of many Kaggle competitions including this one.
3. **Efficiency at scale** — Despite 1.4 million rows and 100,009 features, LightGBM uses histogram-based splitting and leaf-wise tree growth, making training feasible on a single machine.
4. **Built-in regularization** — `subsample`, `colsample_bytree`, and `min_child_samples` prevent overfitting without manual feature selection.

**Key hyperparameters:**
- `n_estimators=1000` — number of boosting rounds (trees)
- `learning_rate=0.05` — step size per tree; smaller = more careful learning, requires more trees
- `num_leaves=63` — controls tree complexity; higher = more expressive but overfitting risk
- `subsample=0.8` — uses 80% of data per tree (adds randomness, prevents overfitting)
- `colsample_bytree=0.8` — uses 80% of features per tree (same purpose)
- `n_jobs=-1` — uses all available CPU cores for parallel training

In [8]:
# Train LightGBM

print("=" * 50)
print("MODEL 2: LIGHTGBM")
print("=" * 50)

lgbm_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,          # use all CPU cores use
    verbose=-1          # training log suppress
)

lgbm_results, lgbm_best_split = train_and_compare_splits(
    model=lgbm_model,
    model_name="LightGBM",
    X=X,
    y=y
)

MODEL 2: LIGHTGBM

Training LightGBM | Split: 70/30
  Train size : 1,037,162
  Val size   : 444,499

  LightGBM | 70/30
  RMSLE          : 0.4660
  RMSE (log)     : 0.4660
  MAE (USD)      : $10.48  ← real price scale
  Train time : 603.6s

Training LightGBM | Split: 80/20
  Train size : 1,185,328
  Val size   : 296,333

  LightGBM | 80/20
  RMSLE          : 0.4650
  RMSE (log)     : 0.4650
  MAE (USD)      : $10.44  ← real price scale
  Train time : 677.4s

  ✅ Best split for LightGBM: 80/20


---
## Step 7 — Final Model Comparison

We compare both models across both splits. The model with the **lowest RMSLE** on the validation set is selected for deployment.

RMSLE is used as the primary metric because it:
- Measures **proportional error** (a 20% mistake on a $10 item is penalized equally to a 20% mistake on a $100 item)
- Is the **official Kaggle metric** for this competition
- Aligns naturally with our `log1p` target transformation

In [9]:
# Final Comparison Table

print("\n")
print("=" * 60)
print("  FINAL MODEL COMPARISON")
print("=" * 60)
print(f"  {'Model':<25} {'Split':<10} {'RMSLE':<10} {'Time'}")
print(f"  {'-'*55}")

for split in ['70/30', '80/20']:
    r_score = ridge_results[split]['score']
    r_time  = ridge_results[split]['train_time']
    l_score = lgbm_results[split]['score']
    l_time  = lgbm_results[split]['train_time']
    
    print(f"  {'Ridge Regression':<25} {split:<10} {r_score:<10.4f} {r_time:.1f}s")
    print(f"  {'LightGBM':<25} {split:<10} {l_score:<10.4f} {l_time:.1f}s")
    print(f"  {'-'*55}")

print(f"\n  Best Ridge  : {ridge_best_split}")
print(f"  Best LightGBM: {lgbm_best_split}")
print("=" * 60)



  FINAL MODEL COMPARISON
  Model                     Split      RMSLE      Time
  -------------------------------------------------------
  Ridge Regression          70/30      0.6246     1.8s
  LightGBM                  70/30      0.4660     603.6s
  -------------------------------------------------------
  Ridge Regression          80/20      0.6248     2.1s
  LightGBM                  80/20      0.4650     677.4s
  -------------------------------------------------------

  Best Ridge  : 70/30
  Best LightGBM: 80/20


---
## Step 8 — Save Best Models

We save both best-performing models to the `models/` directory using `joblib`. These files will be loaded by the FastAPI application in Phase III for real-time inference.

We save both models (not just the winner) so that:
- Ridge serves as a lightweight fallback if LightGBM has resource constraints
- The deployment can be easily switched between models without retraining

In [ ]:
import os
import joblib

os.makedirs('../models', exist_ok=True)

# Best Ridge model save
best_ridge = ridge_results[ridge_best_split]['model']
joblib.dump(best_ridge, '../models/ridge_model.pkl')

# Best LightGBM model save
best_lgbm = lgbm_results[lgbm_best_split]['model']
joblib.dump(best_lgbm, '../models/lgbm_model.pkl')

print("Models saved:")
for f in ['ridge_model.pkl', 'lgbm_model.pkl']:
    size = os.path.getsize(f'../models/{f}') / 1e6
    print(f"  {f:<30} {size:.1f} MB")

Models saved:
  ridge_model.pkl                0.8 MB
  lgbm_model.pkl                 10.2 MB


---
# Phase II Summary — Best Model Selection

### Results

| Model | Split | RMSLE | MAE (USD) | Train Time |
|---|---|---|---|---|
| Ridge Regression | 70/30 | 0.6246 | $14.17 | 1.8s |
| Ridge Regression | 80/20 | 0.6248 | $14.16 | 2.1s |
| LightGBM | 70/30 | 0.4660 | $10.48 | 603s |
| **LightGBM** | **80/20** | **0.4650** | **$10.44** | **677s** |

### ✅ Selected Model: LightGBM (80/20 split)

**LightGBM with 80/20 split is the best-performing model** with RMSLE of **0.4650** — a **25.5% improvement** over Ridge Regression (0.6246).

**Why LightGBM won:**
- Captured non-linear interactions between features (e.g. brand + condition + category combined effects)
- Benefited from the larger training set in 80/20 split — complex models learn more from additional data
- Ridge is limited to linear relationships; price prediction inherently involves complex feature interactions

**Why 80/20 over 70/30 for LightGBM:**
- Gradient boosting models improve with more training data
- 80/20 provides 148,166 additional training rows over 70/30
- RMSLE improvement: 0.4660 → 0.4650 (marginal but consistent)

**For Ridge, 70/30 was marginally better** (0.6246 vs 0.6248) — linear models plateau earlier and don't benefit as much from extra data at this scale.

---

# Project Outcomes

### Problem Solved
Mercari sellers often struggle to price their items competitively. Overpricing leads to unsold listings; underpricing leaves money on the table. This project delivers an **automated price suggestion system** that analyses a product's textual description and attributes to recommend a fair market price.

### Business Value

| Metric | Value | Business Meaning |
|---|---|---|
| Mean Absolute Error | $10.44 | On average, suggestions are within $10.44 of the actual sale price |
| RMSLE | 0.4650 | Proportional error is consistent across price ranges — cheap and expensive items get equally good suggestions |
| Coverage | 1.48M listings | Model trained on real marketplace data across all major categories |

### Impact
- **For sellers:** Reduces pricing uncertainty and time spent researching comparable listings. A new seller with no market knowledge gets an instant, data-driven price recommendation.
- **For Mercari:** Faster listing creation, higher conversion rates, and a more competitive marketplace where items are priced fairly.
- **For buyers:** More consistently priced listings improve trust and reduce negotiation friction.

### Deployed As
The trained LightGBM model is deployed as a **REST API using FastAPI**, containerized with **Docker**, and published to **DockerHub** (`isttiiak/mercari-price-suggestion`). Any seller platform can integrate this API with a single HTTP POST request to get an instant price suggestion.